In [ ]:
import json
import time
from kafka import KafkaProducer
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lpad, concat, lit

# 1. Initialize Kafka Producer
producer = KafkaProducer(
    bootstrap_servers=['kafka:9092'],
    api_version=(3, 0, 0),
    value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8')
)

# 2. Connect Spark with a higher memory limit
spark = SparkSession.builder \
    .appName("ContinuousFlightSimulator") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# 3. Read data from HDFS
print("Loading dedicated stream datasets...")
flights_stream = spark.read.csv("hdfs://namenode:9000/flight_project/raw/flights_stream.csv", header=True, inferSchema=True)
weather_stream = spark.read.csv("hdfs://namenode:9000/flight_project/raw/weather_env_stream.csv", header=True, inferSchema=True)

# 4. Quick Join Logic
hour = (col("crs_dep_time") / 100).cast("int")
minute = col("crs_dep_time") % 100
rounded_hour = when(minute >= 30, hour + 1).otherwise(hour)
rounded_hour = when(rounded_hour == 24, 0).otherwise(rounded_hour)

flights_stream = flights_stream.withColumn(
    "rounded_time", 
    concat(lpad(rounded_hour.cast("string"), 2, "0"), lit("00"))
)
weather_stream = weather_stream.withColumn(
    "time_hhmm", 
    lpad(col("time_hhmm").cast("string"), 4, "0")
)

joined_stream_df = flights_stream.join(
    weather_stream,
    (flights_stream["fl_date"] == weather_stream["fl_date"]) & 
    (flights_stream["origin"] == weather_stream["airport"]) & 
    (flights_stream["rounded_time"] == weather_stream["time_hhmm"]),
    "left"
).drop("rounded_time", weather_stream["fl_date"], "airport", "time_hhmm")

# 5. Safe Infinite Stream using Spark collect chunks (Prevents OOM)
print("🛫 Starting Safe Infinite Flight Stream to Kafka ('flight-stream-raw')...")
print("-" * 50)

try:
    while True:
        # Pull a safe, small batch of 200 rows natively using Spark limit + collect
        batch_rows = joined_stream_df.limit(200).collect()
        
        for row in batch_rows:
            flight_event = row.asDict()
            
            # Send to Kafka
            producer.send('flight-stream-raw', value=flight_event)
            
            carrier = flight_event.get('op_unique_carrier', 'UNK')
            fl_num = flight_event.get('op_carrier_fl_num', '0000')
            origin = flight_event.get('origin', 'UNK')
            
            print(f"[{time.strftime('%X')}] Sent RAW -> Flight {carrier} {fl_num} from {origin}")
            time.sleep(1) # 1 flight per second
            
except KeyboardInterrupt:
    print("\n🛑 Producer stopped manually by user.")
    producer.flush()

Loading dedicated stream datasets...
🛫 Starting Safe Infinite Flight Stream to Kafka ('flight-stream-raw')...
--------------------------------------------------
[03:58:33] Sent RAW -> Flight 9E 5050.0 from ATL
[03:58:34] Sent RAW -> Flight 9E 5054.0 from CHS
[03:58:35] Sent RAW -> Flight 9E 5072.0 from JFK
[03:58:36] Sent RAW -> Flight 9E 4926.0 from LGA
[03:58:37] Sent RAW -> Flight 9E 4938.0 from LGA
[03:58:38] Sent RAW -> Flight 9E 4996.0 from LGA
[03:58:39] Sent RAW -> Flight 9E 4997.0 from LGA
[03:58:40] Sent RAW -> Flight 9E 5003.0 from LGA
[03:58:41] Sent RAW -> Flight 9E 5037.0 from LGA
[03:58:42] Sent RAW -> Flight 9E 4903.0 from MEM
[03:58:43] Sent RAW -> Flight 9E 5011.0 from MEM
[03:58:44] Sent RAW -> Flight 9E 5009.0 from RDU
[03:58:45] Sent RAW -> Flight 9E 4922.0 from SDF
[03:58:46] Sent RAW -> Flight WN 4534.0 from BUR
[03:58:47] Sent RAW -> Flight WN 4465.0 from CVG
[03:58:48] Sent RAW -> Flight WN 4489.0 from LGA
[03:58:49] Sent RAW -> Flight WN 4478.0 from MKE
[03:58